In [0]:
# Assumption : Google Service Account was set on the workspace. Otherwise, follow steps in set_gcp_credentials and databericks_load_data_from_gcs
sc = spark.sparkContext

In [0]:
business_file_path = "gs://dsai694-example/SF_business/filtered_registered_business_sf.tsv"
business_rdd = sc.textFile(business_file_path).map(lambda x: x.split("\t")).map(lambda x : (x[0], x[1:]))

In [0]:
business_rdd.first()

('94123', ['Tournahu George L', '3301 Broderick St', 'San Francisco', 'CA'])

In [0]:
supervisor_file_path = "gs://dsai694-example/SF_business/supervisor_sf.tsv"
supervisor_rdd = sc.textFile(supervisor_file_path).map(lambda x: x.split("\t")).map(lambda x : (x[0], x[1]))

In [0]:
supervisor_rdd.first()

('94102', '8')

In [0]:
supervisor_zip_biz = supervisor_rdd.leftOuterJoin(business_rdd).map(lambda x :(x[1][0], (x[0] , x[1][1][0])))

In [0]:
zip_group = supervisor_zip_biz.groupBy(lambda x : x[1][0])

Save to GCS using `saveAsTextFile()`

In [0]:
supervisor_zip_biz.saveAsTextFile("gs://dsai694-example/SF_business/supervisor_zip_biz")

In [0]:
import pandas as pd
import plotly.express as px
import zipcodes

# Your line: (zip, [records...])
zip_data = zip_group.map(lambda x: (x[0], list(x[1]))).collect()

# One row per zip: list size + coordinates
rows, missing = [], []
for zip_code, items in zip_data:
    match = zipcodes.matching(str(zip_code)[:5])   # [:5] handles ZIP+4 values
    if not match or not match[0]["lat"]:
        missing.append(zip_code)
        continue
    rows.append({
        "zip": zip_code,
        "count": len(items),
        "lat": float(match[0]["lat"]),
        "lon": float(match[0]["long"]),
    })

df = pd.DataFrame(rows)
# print(f"{len(df)} zip codes mapped; no coordinates for: {missing}")

fig = px.scatter_map(
    df,
    lat="lat", lon="lon",
    size="count", color="count",
    hover_name="zip",
    hover_data={"count": True, "lat": False, "lon": False},
    color_continuous_scale="Viridis",
    size_max=40,
    zoom=11,
    center={"lat": 37.76, "lon": -122.44},   # San Francisco
    map_style="open-street-map",
    title="Number of businesses per zip code (San Francisco)",
)
fig.show()

<!doctype html>